# Decision Review & Calibration

Leak detection, tilt analysis, calibration curves.

In [ ]:
import numpy as np
from ipywidgets import interactive, HBox, VBox, Output, IntSlider
from IPython.display import display

from speculator_ev_engine.decisions.logger import Decision, DecisionLogger
from speculator_ev_engine.decisions.leaks import detect_leaks, detect_tilt
from speculator_ev_engine.decisions.calibration import full_calibration
from speculator_ev_engine.decisions.review import review_decisions, format_review
from speculator_ev_engine.ui.core.charts import ev_vs_outcome_scatter, tilt_detection_time_series, calibration_curve

In [ ]:
# Generate sample decisions for analysis
rng = np.random.default_rng(42)
logger = DecisionLogger()
for i in range(200):
    p = rng.uniform(0.3, 0.7)
    ev = p * 1.0 - (1-p) * 1.0
    stake = rng.uniform(10, 100)
    outcome = float(rng.choice([1.0, -1.0], p=[p, 1-p]))
    domain = rng.choice(['poker', 'sports', 'markets'])
    d = Decision(decision=f'dec_{i}', p_estimate=float(p), ev_estimate=float(ev),
                 stake=float(stake), outcome=outcome, domain=str(domain))
    logger.log(d)

decisions = logger.query(limit=500)
print(f'Loaded {len(decisions)} decisions')

In [ ]:
# EV vs Outcome scatter
evs = np.array([d.ev_estimate for d in decisions])
outs = np.array([d.outcome or 0.0 for d in decisions])
fig = ev_vs_outcome_scatter(evs, outs)
display(fig)
plt.close(fig)

In [ ]:
# Calibration curve
forecasts = np.array([d.p_estimate for d in decisions])
outcomes = np.array([d.outcome or 0.0 for d in decisions])
# Binary outcomes: win if outcome > 0
binary_outcomes = (outcomes > 0).astype(float)
cal = full_calibration(forecasts, binary_outcomes, n_bins=10)
fig2 = calibration_curve(cal.bin_centers, cal.bin_frequencies, cal.bin_counts, cal.ece)
display(fig2)
plt.close(fig2)

In [ ]:
# Tilt detection
tilts = detect_tilt(decisions)
if tilts:
    stakes = np.array([d.stake for d in decisions])
    loss_indicators = np.array([(d.outcome or 0) < 0 for d in decisions], dtype=float)
    # Running average of recent loss context
    window = 10
    smoothed_losses = np.convolve(loss_indicators, np.ones(window)/window, mode='same')
    fig3 = tilt_detection_time_series(stakes, smoothed_losses)
    display(fig3)
    plt.close(fig3)
    for t in tilts:
        print(f'Tilt at decision {t.start_index}: severity={t.severity:.1f} — {t.pattern}')
else:
    print('No tilt patterns detected')